# CARREFOUR — Sell-In / Sell-Out en KG uniquement

Objectif : comparer le Sell-In CARREFOUR au Sell-Out Nielsen en utilisant **uniquement les kilogrammes**.

Règle de périmètre :
- `CARREFOUR_IN["CODE EAN"]`
- `Nielsen["UPC"]`
- seuls les codes présents dans **les deux fichiers** sont conservés.

Règle d'unité :
- Sell-In = colonne `CARREFOUR` de CARREFOUR_IN, en kg
- Sell-Out = colonne `CARREFOUR` de Nielsen, en kg
- la colonne `CARREFOUR en UVC` reste informative uniquement et n'est pas utilisée pour classifier stockage / déstockage.


## 1. Imports et paramètres

Les seuils sont paramétrables. Par défaut :
- accumulation si `Sell-In / Sell-Out > 1,15` ;
- déstockage si `Sell-In / Sell-Out < 0,85` ;
- baisse forte des achats si le Sell-In est inférieur d'au moins 30 % à la médiane des 3 périodes précédentes ;
- un régime est confirmé après 2 périodes Nielsen consécutives.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown
import plotly.io as pio

pio.renderers.default = "notebook_connected"  # ou "iframe"

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

STOCK_TOLERANCE = 0.15
PURCHASE_DROP_THRESHOLD = 0.30
MIN_CONSECUTIVE_PERIODS = 2
BASELINE_PERIODS = 3
MAX_LAG_WEEKS = 8

# IMPORTANT : le fichier Nielsen ne nomme pas explicitement l'unité de la colonne CARREFOUR.
# Par défaut nous l'interprétons comme UVC car POIDS DE BASE permet ensuite une conversion kg.
# Si votre export Nielsen est déjà en kg, passer ce paramètre à False.
NIELSEN_VALUES_ARE_UVC = True


def resolve_file(filename):
    candidates = [
        Path.cwd() / filename,
        Path('/mnt/data') / filename,
        Path('/mnt/data/_test_carrefour_nb') / filename,
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Fichier introuvable : {filename}. Placez-le à côté du notebook.")

PATH_SELL_IN = resolve_file('CARREFOUR_IN.xlsx')
PATH_SELL_OUT = resolve_file('Nielsen_SELL_OUT_0826.xlsx')

print('Sell-In :', PATH_SELL_IN)
print('Sell-Out:', PATH_SELL_OUT)

Sell-In : c:\Users\andrea.leylavergne\IA Projects\SELL_IN_SELL_OUT\CARREFOUR_IN.xlsx
Sell-Out: c:\Users\andrea.leylavergne\IA Projects\SELL_IN_SELL_OUT\Nielsen_SELL_OUT_0826.xlsx


# PARTIE A — Préparer le Sell-In CARREFOUR

## 2. Charger le fichier CARREFOUR

Dans le fichier fourni :
- `CARREFOUR` = volume livré en kg ;
- `CARREFOUR en UVC` = unités livrées ;
- `CODE EAN` = produit ;
- `DATE` = date hebdomadaire utilisée pour construire les fenêtres Nielsen de 4 semaines.

In [2]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    df_in_raw = pd.read_excel(PATH_SELL_IN, header=1)

df_in_raw.columns = (
    df_in_raw.columns.astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)
)

rename_in = {
    'CODE EAN': 'EAN_SOURCE',
    'CARREFOUR': 'SELL_IN_KG',
    'CARREFOUR en UVC': 'SELL_IN_UVC',
}
df_in_raw = df_in_raw.rename(columns=rename_in)

required_in = {'DATE','EAN_SOURCE','SELL_IN_KG','SELL_IN_UVC'}
missing = required_in.difference(df_in_raw.columns)
assert not missing, f"Colonnes Sell-In manquantes : {sorted(missing)}"

def normalize_product_code(series):
    # Rend comparable un EAN/UPC lu comme entier, float ou texte.
    # Le retrait des zéros initiaux permet aussi de rapprocher un UPC-A 12 chiffres
    # d'un EAN-13 équivalent préfixé par 0.
    s = (
        series.astype(str)
        .str.strip()
        .str.replace(r'\.0$', '', regex=True)
        .str.replace(r'\D', '', regex=True)
        .str.lstrip('0')
    )
    return s.replace('', pd.NA)

df_in_raw['DATE'] = pd.to_datetime(df_in_raw['DATE'], errors='coerce')
df_in_raw['EAN'] = normalize_product_code(df_in_raw['EAN_SOURCE'])
df_in_raw['SELL_IN_KG'] = pd.to_numeric(df_in_raw['SELL_IN_KG'], errors='coerce').fillna(0)
df_in_raw['SELL_IN_UVC'] = pd.to_numeric(df_in_raw['SELL_IN_UVC'], errors='coerce').fillna(0)

df_in_raw = (
    df_in_raw
    .dropna(subset=['DATE','EAN'])
    .sort_values(['DATE','EAN'])
    .reset_index(drop=True)
)

print('Lignes Sell-In :', len(df_in_raw))
print('CODE EAN distincts :', df_in_raw['EAN'].nunique())
print('Période :', df_in_raw['DATE'].min().date(), '->', df_in_raw['DATE'].max().date())
display(df_in_raw[['DATE','EAN_SOURCE','EAN','SELL_IN_KG','SELL_IN_UVC']].head())


Lignes Sell-In : 33227
CODE EAN distincts : 96
Période : 2023-01-22 -> 2026-07-12


,DATE,EAN_SOURCE,EAN,SELL_IN_KG,SELL_IN_UVC
0,2023-01-22,3497911101129,3497911101129,950.0,7600
1,2023-01-22,3497911101129,3497911101129,2200.0,17600
2,2023-01-22,3497911105127,3497911105127,600.0,4800
3,2023-01-22,3497911105127,3497911105127,2500.0,20000
4,2023-01-22,3497911119124,3497911119124,200.0,1600


## 3. Contrôle kg ↔ UVC du Sell-In

Pour chaque EAN, le ratio `kg / UVC` doit être relativement stable si le conditionnement ne change pas. Ce contrôle aide à sécuriser les unités avant comparaison avec Nielsen.

In [3]:
weight_check = (
    df_in_raw[df_in_raw['SELL_IN_UVC'] > 0]
    .assign(POIDS_IMPLICITE_G=lambda x: x['SELL_IN_KG'] * 1000 / x['SELL_IN_UVC'])
    .groupby('EAN', as_index=False)
    .agg(
        NB_LIGNES=('DATE','size'),
        POIDS_IMPLICITE_G_MEDIAN=('POIDS_IMPLICITE_G','median'),
        POIDS_IMPLICITE_G_MIN=('POIDS_IMPLICITE_G','min'),
        POIDS_IMPLICITE_G_MAX=('POIDS_IMPLICITE_G','max'),
    )
    .sort_values('NB_LIGNES', ascending=False)
)
display(weight_check.head(15).round(2))

,EAN,NB_LIGNES,POIDS_IMPLICITE_G_MEDIAN,POIDS_IMPLICITE_G_MIN,POIDS_IMPLICITE_G_MAX
26,3497917000518,883,125.0,125.0,125.0
30,3497917000907,879,125.0,125.0,125.0
3,3497911105127,864,125.0,125.0,125.0
49,3497917002741,861,125.0,125.0,125.0
0,3497911101129,854,125.0,125.0,125.0
10,3497911219251,816,250.0,250.0,250.0
8,3497911131126,802,250.0,250.0,250.0
56,3497917003168,801,125.0,125.0,125.0
19,3497917000136,796,250.0,250.0,250.0
46,3497917002352,793,125.0,125.0,125.0


# PARTIE B — Préparer le Sell-Out Nielsen

## 4. Rapprocher `CODE EAN` et `UPC` par intersection

`CODE EAN` dans **CARREFOUR_IN** et `UPC` dans **Nielsen** sont utilisés comme la même clé produit.

La règle est stricte :

**EAN communs = CODE EAN présents dans CARREFOUR_IN ∩ UPC présents dans Nielsen**

Ensuite, **les deux tables sont filtrées sur ces seuls codes communs** avant tout calcul Sell-In / Sell-Out.

Aucun UPC Nielsen absent de CARREFOUR_IN n'est utilisé, et aucun CODE EAN CARREFOUR absent de Nielsen n'est utilisé.


In [4]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    df_out_raw = pd.read_excel(PATH_SELL_OUT, sheet_name='Feuil4')

df_out_raw.columns = (
    df_out_raw.columns.astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)
)

required_out = {'Periods','UPC','CARREFOUR'}
missing = required_out.difference(df_out_raw.columns)
assert not missing, f"Colonnes Nielsen manquantes : {sorted(missing)}"

date_txt = df_out_raw['Periods'].astype(str).str.extract(r'(\d{2}/\d{2}/\d{2})$')[0]
df_out_raw['DATE'] = pd.to_datetime(date_txt, format='%d/%m/%y', errors='coerce')

# Même normalisation que CODE EAN
df_out_raw['UPC_SOURCE'] = df_out_raw['UPC']
df_out_raw['EAN'] = normalize_product_code(df_out_raw['UPC'])

df_out_raw['CARREFOUR'] = pd.to_numeric(
    df_out_raw['CARREFOUR'], errors='coerce'
).fillna(0)

if 'POIDS DE BASE' in df_out_raw.columns:
    df_out_raw['POIDS_DE_BASE_G'] = pd.to_numeric(
        df_out_raw['POIDS DE BASE'], errors='coerce'
    )
else:
    df_out_raw['POIDS_DE_BASE_G'] = np.nan

# ============================================================
# INTERSECTION STRICTE DES REFERENCES PRODUITS
# ============================================================
ean_sell_in = set(df_in_raw['EAN'].dropna().unique())
upc_nielsen = set(df_out_raw['EAN'].dropna().unique())

matched_eans = sorted(ean_sell_in.intersection(upc_nielsen))

print(f"CODE EAN distincts dans CARREFOUR_IN : {len(ean_sell_in)}")
print(f"UPC distincts dans Nielsen          : {len(upc_nielsen)}")
print(f"Références présentes dans LES DEUX  : {len(matched_eans)}")

if len(matched_eans) == 0:
    raise ValueError(
        "Aucune référence commune entre CARREFOUR_IN[CODE EAN] et Nielsen[UPC]. "
        "Vérifier le fichier Nielsen et le format des codes."
    )

# On filtre LES DEUX bases sur la même intersection
df_in_common = df_in_raw[df_in_raw['EAN'].isin(matched_eans)].copy()
df_out_common = df_out_raw[df_out_raw['EAN'].isin(matched_eans)].copy()

assert set(df_in_common['EAN'].unique()) == set(matched_eans)
assert set(df_out_common['EAN'].unique()) == set(matched_eans)

coverage_summary = pd.DataFrame({
    'INDICATEUR': [
        'CODE EAN distincts CARREFOUR_IN',
        'UPC distincts Nielsen',
        'EAN / UPC communs utilisés',
        'Périodes Nielsen'
    ],
    'VALEUR': [
        len(ean_sell_in),
        len(upc_nielsen),
        len(matched_eans),
        df_out_common['DATE'].nunique()
    ]
})
display(coverage_summary)

# Liste exacte des produits communs
cols_products = ['EAN']
if 'RECETTE' in df_out_common.columns:
    cols_products.append('RECETTE')
if 'POIDS_DE_BASE_G' in df_out_common.columns:
    cols_products.append('POIDS_DE_BASE_G')

common_products = (
    df_out_common[cols_products]
    .drop_duplicates(subset=['EAN'])
    .sort_values('EAN')
    .reset_index(drop=True)
)

display(common_products)


CODE EAN distincts dans CARREFOUR_IN : 96
UPC distincts dans Nielsen          : 95
Références présentes dans LES DEUX  : 66


,INDICATEUR,VALEUR
0,CODE EAN distincts CARREFOUR_IN,96
1,UPC distincts Nielsen,95
2,EAN / UPC communs utilisés,66
3,Périodes Nielsen,45


,EAN,RECETTE,POIDS_DE_BASE_G
0,3497911101129,POULET BRAISE,125
1,3497911102126,BARBECUE,125
2,3497911105127,SEL ET VINAIGRE,125
3,3497911119124,SEL DE GUERANDE,125
4,3497911119148,SEL DE GUERANDE,25
...,...,...,...
61,3497917004240,BRIE TRUFFE,125
62,3497917004257,BBQ & JALAPENO,125
63,3497917004264,BBQ MEXICAINE,125
64,3497917004271,BBQ & FLAMBE AU WHISKY,125


## 5. Contrôle du périmètre commun

À partir d'ici, **tous les calculs utilisent uniquement `matched_eans`**, c'est-à-dire les références réellement présentes dans les deux fichiers.

La part du Sell-In représentée par ce périmètre est affichée uniquement comme contrôle de couverture ; elle ne sert pas à exclure d'autres codes au-delà de la règle d'intersection.


In [5]:
# Dates Nielsen pour lesquelles 4 semaines Sell-In sont disponibles
nielsen_dates = sorted(df_out_common['DATE'].dropna().unique())
used_week_dates = set()
valid_end_dates = []

for d0 in nielsen_dates:
    d = pd.Timestamp(d0)
    expected = [d - pd.Timedelta(weeks=i) for i in range(4)]
    if df_in_common[df_in_common['DATE'].isin(expected)]['DATE'].nunique() == 4:
        valid_end_dates.append(d)
        used_week_dates.update(expected)

scope_all_in = df_in_raw[df_in_raw['DATE'].isin(used_week_dates)].copy()
scope_common_in = df_in_common[df_in_common['DATE'].isin(used_week_dates)].copy()

coverage_kg = (
    scope_common_in['SELL_IN_KG'].sum() / scope_all_in['SELL_IN_KG'].sum()
    if scope_all_in['SELL_IN_KG'].sum() else np.nan
)
coverage_uvc = (
    scope_common_in['SELL_IN_UVC'].sum() / scope_all_in['SELL_IN_UVC'].sum()
    if scope_all_in['SELL_IN_UVC'].sum() else np.nan
)

coverage_volume = pd.DataFrame({
    'MESURE': [
        'Part du Sell-In kg appartenant aux EAN/UPC communs',
        'Part du Sell-In UVC appartenant aux EAN/UPC communs'
    ],
    'TAUX': [coverage_kg, coverage_uvc]
})
display(coverage_volume.style.format({'TAUX':'{:.1%}'}))


,MESURE,TAUX
0,Part du Sell-In kg appartenant aux EAN/UPC communs,95.2%
1,Part du Sell-In UVC appartenant aux EAN/UPC communs,97.1%


# PARTIE C — Construire un comparatif EAN × période Nielsen strictement comparable

## 6. Transformer le Sell-Out dans les deux unités

Le calcul conserve **UVC et kg**.

- Si `NIELSEN_VALUES_ARE_UVC = True`, la valeur Nielsen CARREFOUR est interprétée comme un nombre d'unités consommateurs puis convertie en kg via `POIDS DE BASE`.
- Si votre export Nielsen est déjà en kg, passer le paramètre à `False`.

In [6]:
# Sell-Out Nielsen : la colonne CARREFOUR est déjà en KG
df_out = df_out_common.copy()

df_out['SELL_OUT_KG'] = pd.to_numeric(
    df_out['CARREFOUR'], errors='coerce'
).fillna(0)

sell_out_product_4w = (
    df_out.groupby(['DATE','EAN'], as_index=False)
    .agg(
        SELL_OUT_KG=('SELL_OUT_KG','sum'),
        RECETTE=('RECETTE','first') if 'RECETTE' in df_out.columns else ('EAN','first')
    )
    .sort_values(['DATE','EAN'])
)

display(sell_out_product_4w.head())


,DATE,EAN,SELL_OUT_KG,RECETTE
0,2023-02-26,3497911101129,21699,POULET BRAISE
1,2023-02-26,3497911102126,1314,BARBECUE
2,2023-02-26,3497911105127,24297,SEL ET VINAIGRE
3,2023-02-26,3497911119124,3876,SEL DE GUERANDE
4,2023-02-26,3497911119148,0,SEL DE GUERANDE


## 7. Agréger le Sell-In sur les mêmes fenêtres de 4 semaines

In [7]:
product_rows = []

for end_date in sorted(sell_out_product_4w['DATE'].dropna().unique()):
    end_date = pd.Timestamp(end_date)
    expected = [end_date - pd.Timedelta(weeks=i) for i in range(4)]

    # Sell-In uniquement sur les EAN/UPC communs
    w = df_in_common[df_in_common['DATE'].isin(expected)]

    if w['DATE'].nunique() != 4:
        continue

    agg = (
        w.groupby('EAN', as_index=False)
        .agg(
            SELL_IN_KG=('SELL_IN_KG','sum'),
            SELL_IN_UVC=('SELL_IN_UVC','sum')   # informatif seulement
        )
    )
    agg['DATE'] = end_date
    product_rows.append(agg)

sell_in_product_4w = pd.concat(product_rows, ignore_index=True)

# INNER JOIN strict sur DATE + EAN
df_product = (
    sell_in_product_4w
    .merge(
        sell_out_product_4w,
        on=['DATE','EAN'],
        how='inner',
        validate='one_to_one'
    )
    .sort_values(['EAN','DATE'])
    .reset_index(drop=True)
)

print('EAN/UPC effectivement comparés :', df_product['EAN'].nunique())
display(df_product.head(10).round(2))


EAN/UPC effectivement comparés : 65


C:\Users\andrea.leylavergne\AppData\Local\Temp\ipykernel_7088\2566751326.py:39: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(df_product.head(10).round(2))


,EAN,SELL_IN_KG,SELL_IN_UVC,DATE,SELL_OUT_KG,RECETTE
0,3497911101129,21500.0,172000,2023-02-26,21699,POULET BRAISE
1,3497911101129,20100.0,160800,2023-03-26,21393,POULET BRAISE
2,3497911101129,35050.0,280400,2023-04-23,22591,POULET BRAISE
3,3497911101129,22550.0,180400,2023-05-21,23746,POULET BRAISE
4,3497911101129,28500.0,228000,2023-06-18,24448,POULET BRAISE
5,3497911101129,30910.0,247280,2023-07-16,24915,POULET BRAISE
6,3497911101129,20550.0,164400,2023-08-13,25783,POULET BRAISE
7,3497911101129,27700.0,221600,2023-09-10,25616,POULET BRAISE
8,3497911101129,19600.0,156800,2023-10-08,24627,POULET BRAISE
9,3497911101129,17850.0,142800,2023-11-05,19615,POULET BRAISE


# PARTIE D — Détecter accumulation, déstockage et ralentissement des achats

## 8. Indicateurs au niveau EAN

Les ratios sont calculés en UVC. Quand les poids sont cohérents, le ratio kg est identique.

In [8]:
df_product['GAP_KG'] = df_product['SELL_IN_KG'] - df_product['SELL_OUT_KG']
df_product['RATIO_IN_OUT'] = (
    df_product['SELL_IN_KG'] /
    df_product['SELL_OUT_KG'].replace(0, np.nan)
)
df_product['GAP_PCT_OUT'] = (
    df_product['GAP_KG'] /
    df_product['SELL_OUT_KG'].replace(0, np.nan) * 100
)

upper_stock = 1 + STOCK_TOLERANCE
lower_stock = 1 - STOCK_TOLERANCE

df_product['REGIME'] = np.select(
    [
        df_product['RATIO_IN_OUT'] > upper_stock,
        df_product['RATIO_IN_OUT'] < lower_stock
    ],
    ['ACCUMULATION_STOCK', 'DESTOCKAGE'],
    default='EQUILIBRE'
)

# Baseline d'achat passée, en kg
baseline = (
    df_product.groupby('EAN')['SELL_IN_KG']
    .transform(
        lambda s: s.shift(1).rolling(
            BASELINE_PERIODS,
            min_periods=2
        ).median()
    )
)

df_product['SELL_IN_BASELINE_PREV3_KG'] = baseline
df_product['ACHAT_REL_BASELINE'] = (
    df_product['SELL_IN_KG'] /
    baseline.replace(0, np.nan)
)
df_product['BAISSE_ACHATS_PCT'] = (
    1 - df_product['ACHAT_REL_BASELINE']
) * 100

df_product['BAISSE_ACHATS_FORTE'] = (
    df_product['ACHAT_REL_BASELINE']
    <= (1 - PURCHASE_DROP_THRESHOLD)
)

cols = [
    'DATE','EAN','RECETTE',
    'SELL_IN_KG','SELL_OUT_KG',
    'GAP_KG','RATIO_IN_OUT','REGIME',
    'BAISSE_ACHATS_FORTE','BAISSE_ACHATS_PCT'
]

display(df_product[cols].tail(20).round(2))


C:\Users\andrea.leylavergne\AppData\Local\Temp\ipykernel_7088\3444781424.py:55: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(df_product[cols].tail(20).round(2))


,DATE,EAN,RECETTE,SELL_IN_KG,SELL_OUT_KG,GAP_KG,RATIO_IN_OUT,REGIME,BAISSE_ACHATS_FORTE,BAISSE_ACHATS_PCT
2078,2026-04-19,3497917004240,BRIE TRUFFE,2628.75,1813,815.75,1.45,ACCUMULATION_STOCK,False,NaN
2079,2026-05-17,3497917004240,BRIE TRUFFE,5008.75,3399,1609.75,1.47,ACCUMULATION_STOCK,False,-193.02
2080,2026-06-14,3497917004240,BRIE TRUFFE,4391.25,4937,-545.75,0.89,EQUILIBRE,False,-67.05
2081,2026-07-12,3497917004240,BRIE TRUFFE,7100.00,4981,2119.00,1.43,ACCUMULATION_STOCK,False,-61.69
2082,2026-04-19,3497917004257,BBQ & JALAPENO,6073.75,728,5345.75,8.34,ACCUMULATION_STOCK,False,NaN
2083,2026-05-17,3497917004257,BBQ & JALAPENO,708.75,5775,-5066.25,0.12,DESTOCKAGE,False,NaN
2084,2026-06-14,3497917004257,BBQ & JALAPENO,40.00,2607,-2567.00,0.02,DESTOCKAGE,True,98.82
2085,2026-07-12,3497917004257,BBQ & JALAPENO,38.75,1237,-1198.25,0.03,DESTOCKAGE,True,94.53
2086,2026-03-22,3497917004264,BBQ MEXICAINE,2970.00,26,2944.00,114.23,ACCUMULATION_STOCK,False,NaN
2087,2026-04-19,3497917004264,BBQ MEXICAINE,7190.00,5164,2026.00,1.39,ACCUMULATION_STOCK,False,NaN


## 9. Vue agrégée sur le périmètre réellement couvert

Si plusieurs EAN Nielsen sont ajoutés plus tard, cette cellule les agrégera automatiquement. Avec le fichier actuel, elle correspond au seul EAN couvert.

In [9]:
df_compare = (
    df_product.groupby('DATE', as_index=False)
    .agg(
        SELL_IN_KG=('SELL_IN_KG','sum'),
        SELL_OUT_KG=('SELL_OUT_KG','sum'),
        NB_EAN_COUVERTS=('EAN','nunique')
    )
    .sort_values('DATE')
    .reset_index(drop=True)
)

df_compare['GAP_KG'] = (
    df_compare['SELL_IN_KG'] -
    df_compare['SELL_OUT_KG']
)

df_compare['RATIO_IN_OUT'] = (
    df_compare['SELL_IN_KG'] /
    df_compare['SELL_OUT_KG'].replace(0, np.nan)
)

df_compare['REGIME'] = np.select(
    [
        df_compare['RATIO_IN_OUT'] > upper_stock,
        df_compare['RATIO_IN_OUT'] < lower_stock
    ],
    ['ACCUMULATION_STOCK','DESTOCKAGE'],
    default='EQUILIBRE'
)

# Proxy de stock cumulé, en kg
df_compare['STOCK_PROXY_CUM_KG'] = (
    df_compare['GAP_KG'].cumsum()
)

# Baseline Sell-In passée, en kg
base = (
    df_compare['SELL_IN_KG']
    .shift(1)
    .rolling(BASELINE_PERIODS, min_periods=2)
    .median()
)

df_compare['SELL_IN_BASELINE_PREV3_KG'] = base
df_compare['ACHAT_REL_BASELINE'] = (
    df_compare['SELL_IN_KG'] /
    base.replace(0, np.nan)
)
df_compare['BAISSE_ACHATS_FORTE'] = (
    df_compare['ACHAT_REL_BASELINE']
    <= (1 - PURCHASE_DROP_THRESHOLD)
)
df_compare['BAISSE_ACHATS_PCT'] = (
    1 - df_compare['ACHAT_REL_BASELINE']
) * 100

display(df_compare.tail(15).round(2))


C:\Users\andrea.leylavergne\AppData\Local\Temp\ipykernel_7088\655063432.py:57: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(df_compare.tail(15).round(2))


,DATE,SELL_IN_KG,SELL_OUT_KG,NB_EAN_COUVERTS,GAP_KG,RATIO_IN_OUT,REGIME,STOCK_PROXY_CUM_KG,SELL_IN_BASELINE_PREV3_KG,ACHAT_REL_BASELINE,BAISSE_ACHATS_FORTE,BAISSE_ACHATS_PCT
30,2025-06-15,512413.70,391957,48,120456.70,1.31,ACCUMULATION_STOCK,652258.95,430161.30,1.19,False,-19.12
31,2025-07-13,414223.35,417723,48,-3499.65,0.99,EQUILIBRE,648759.30,436176.70,0.95,False,5.03
32,2025-08-10,486330.90,417621,47,68709.90,1.16,ACCUMULATION_STOCK,717469.20,414223.35,1.17,False,-17.41
33,2025-09-07,281695.80,390322,47,-108626.20,0.72,DESTOCKAGE,608843.00,486330.90,0.58,True,42.08
34,2025-10-05,265897.10,333766,48,-67868.90,0.80,DESTOCKAGE,540974.10,414223.35,0.64,True,35.81
35,2025-11-02,221040.25,334146,47,-113105.75,0.66,DESTOCKAGE,427868.35,281695.80,0.78,False,21.53
36,2025-11-30,403463.65,313669,48,89794.65,1.29,ACCUMULATION_STOCK,517663.00,265897.10,1.52,False,-51.74
37,2025-12-28,461053.40,324090,47,136963.40,1.42,ACCUMULATION_STOCK,654626.40,265897.10,1.73,False,-73.40
38,2026-01-25,320011.30,319275,47,736.30,1.00,EQUILIBRE,655362.70,403463.65,0.79,False,20.68
39,2026-02-22,383867.75,335742,48,48125.75,1.14,EQUILIBRE,703488.45,403463.65,0.95,False,4.86


## 10. Courbe Sell-In / Sell-Out — UVC comparables

In [10]:
import os

# 1. Création et affichage du graphique Plotly
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df_compare["DATE"],
        y=df_compare["SELL_IN_KG"],
        mode="lines+markers",
        name="Sell-In KG — 4 semaines",
    )
)
fig.add_trace(
    go.Scatter(
        x=df_compare["DATE"],
        y=df_compare["SELL_OUT_KG"],
        mode="lines+markers",
        name="Sell-Out Nielsen KG — 4 semaines",
    )
)
fig.update_layout(
    title="CARREFOUR — Sell-In vs Sell-Out sur le périmètre EAN réellement couvert",
    template="plotly_white",
    height=560,
    xaxis_title="Fin de période Nielsen",
    yaxis_title="KG",
    hovermode="x unified",
)
fig.show()

# 2. Sauvegarde de l'image pour Streamlit
os.makedirs("images", exist_ok=True)
fig.write_image(
    "images/courbe_sell_in_out.png", width=1200, height=600, scale=2
)

## 11. Écart d'approvisionnement et proxy de stock

`Sell-In − Sell-Out` est un **flux net**. Son cumul est un **proxy de variation de stock**, pas un stock physique réel : le stock initial reste inconnu.

In [11]:
import os

# 1. Premier graphique : Flux net d'approvisionnement
fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=df_compare["DATE"],
        y=df_compare["GAP_KG"],
        name="Sell-In − Sell-Out (KG)",
    )
)
fig.add_hline(y=0, line_dash="dash", annotation_text="Équilibre")
fig.update_layout(
    title="CARREFOUR — Flux net d’approvisionnement sur le périmètre couvert",
    template="plotly_white",
    height=500,
    yaxis_title="KG",
)
fig.show()

# 2. Deuxième graphique : Variation cumulée (Proxy de stock)
fig2 = px.line(
    df_compare,
    x="DATE",
    y="STOCK_PROXY_CUM_KG",
    markers=True,
    title="CARREFOUR — Variation cumulée théorique du stock (proxy, KG)",
)
fig2.add_hline(y=0, line_dash="dash")
fig2.update_layout(
    template="plotly_white",
    height=500,
    yaxis_title="Cumul Sell-In − Sell-Out (KG)",
)
fig2.show()

# 3. Sauvegarde des DEUX images pour Streamlit
os.makedirs("images", exist_ok=True)

# Sauvegarde du 1er graphique (Barres / Flux net)
fig.write_image("images/flux_net_gap.png", width=1200, height=600, scale=2)

# Sauvegarde du 2ème graphique (Ligne / Proxy de stock)
fig2.write_image("images/proxy_stock.png", width=1200, height=600, scale=2)

## 12. Construire les épisodes confirmés

In [12]:
def build_episodes(df, state_col='REGIME'):
    temp = df.copy().sort_values('DATE').reset_index(drop=True)
    temp['_GROUP'] = temp[state_col].ne(
        temp[state_col].shift()
    ).cumsum()

    episodes = (
        temp.groupby('_GROUP', as_index=False)
        .agg(
            REGIME=(state_col,'first'),
            DEBUT_SIGNAL=('DATE','min'),
            FIN=('DATE','max'),
            NB_PERIODES=('DATE','size'),
            SELL_IN_CUM_KG=('SELL_IN_KG','sum'),
            SELL_OUT_CUM_KG=('SELL_OUT_KG','sum'),
            GAP_CUM_KG=('GAP_KG','sum'),
            RATIO_IN_OUT_MOYEN=('RATIO_IN_OUT','mean')
        )
    )

    episodes['CONFIRME'] = (
        (episodes['REGIME'] != 'EQUILIBRE')
        & (episodes['NB_PERIODES'] >= MIN_CONSECUTIVE_PERIODS)
    )

    episodes['DATE_CONFIRMATION'] = (
        episodes['DEBUT_SIGNAL']
        + pd.to_timedelta(
            (MIN_CONSECUTIVE_PERIODS - 1) * 28,
            unit='D'
        )
    )

    return episodes

episodes = build_episodes(df_compare)
episodes_confirmes = episodes[
    episodes['CONFIRME']
].copy()

display(episodes_confirmes.round(2))


C:\Users\andrea.leylavergne\AppData\Local\Temp\ipykernel_7088\1830077603.py:41: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(episodes_confirmes.round(2))


,_GROUP,REGIME,DEBUT_SIGNAL,FIN,NB_PERIODES,SELL_IN_CUM_KG,SELL_OUT_CUM_KG,GAP_CUM_KG,RATIO_IN_OUT_MOYEN,CONFIRME,DATE_CONFIRMATION
11,12,ACCUMULATION_STOCK,2025-03-23,2025-04-20,2,866338.00,667869,198469.00,1.31,True,2025-04-20
16,17,DESTOCKAGE,2025-09-07,2025-11-02,3,768633.15,1058234,-289600.85,0.73,True,2025-10-05
17,18,ACCUMULATION_STOCK,2025-11-30,2025-12-28,2,864517.05,637759,226758.05,1.35,True,2025-12-28


# PARTIE E — « quand arrête-t-il d'acheter après avoir stocké ? »

## 13. Repérer une transition accumulation → ralentissement

Nous considérons un **basculement achat → utilisation du stock** lorsqu'une période de déstockage / forte baisse d'achat intervient dans les deux périodes qui suivent une phase d'accumulation.

Ce n'est pas une preuve de stock physique, mais un signal comportemental utile à investiguer.

In [13]:
df_transition = df_compare.copy()

df_transition['ACCUM_PREV1'] = (
    df_transition['REGIME']
    .shift(1)
    .eq('ACCUMULATION_STOCK')
)

df_transition['ACCUM_PREV2'] = (
    df_transition['REGIME']
    .shift(2)
    .eq('ACCUMULATION_STOCK')
)

df_transition['TRANSITION_ACHAT_VERS_STOCK'] = (
    (
        df_transition['REGIME'].eq('DESTOCKAGE')
        | df_transition['BAISSE_ACHATS_FORTE']
    )
    & (
        df_transition['ACCUM_PREV1']
        | df_transition['ACCUM_PREV2']
    )
)

transitions = df_transition[
    df_transition['TRANSITION_ACHAT_VERS_STOCK']
][[
    'DATE',
    'SELL_IN_KG',
    'SELL_OUT_KG',
    'GAP_KG',
    'RATIO_IN_OUT',
    'SELL_IN_BASELINE_PREV3_KG',
    'ACHAT_REL_BASELINE',
    'BAISSE_ACHATS_FORTE',
    'REGIME'
]].copy()

display(transitions.round(2))

if not transitions.empty:
    print('Dates candidates de basculement achat → utilisation du stock :')
    for d in transitions['DATE']:
        print(' -', d.strftime('%d/%m/%Y'))
else:
    print('Aucune transition répondant à la règle actuelle.')


C:\Users\andrea.leylavergne\AppData\Local\Temp\ipykernel_7088\4032689042.py:40: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(transitions.round(2))


,DATE,SELL_IN_KG,SELL_OUT_KG,GAP_KG,RATIO_IN_OUT,SELL_IN_BASELINE_PREV3_KG,ACHAT_REL_BASELINE,BAISSE_ACHATS_FORTE,REGIME
11,2023-12-31,130254.1,232694,-102439.9,0.56,173537.00,0.75,False,DESTOCKAGE
33,2025-09-07,281695.8,390322,-108626.2,0.72,486330.90,0.58,True,DESTOCKAGE
34,2025-10-05,265897.1,333766,-67868.9,0.80,414223.35,0.64,True,DESTOCKAGE


Dates candidates de basculement achat → utilisation du stock :
 - 31/12/2023
 - 07/09/2025
 - 05/10/2025


## 14. Toutes les fortes baisses d'achat

Cette table répond directement à « quand CARREFOUR réduit fortement ses commandes ? », indépendamment de la présence d'une accumulation juste avant.

In [14]:
purchase_drops = df_compare[
    df_compare['BAISSE_ACHATS_FORTE']
][[
    'DATE',
    'SELL_IN_KG',
    'SELL_OUT_KG',
    'SELL_IN_BASELINE_PREV3_KG',
    'ACHAT_REL_BASELINE',
    'BAISSE_ACHATS_PCT',
    'GAP_KG',
    'REGIME'
]].copy()

display(purchase_drops.round(2))


C:\Users\andrea.leylavergne\AppData\Local\Temp\ipykernel_7088\2047559784.py:14: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(purchase_drops.round(2))


,DATE,SELL_IN_KG,SELL_OUT_KG,SELL_IN_BASELINE_PREV3_KG,ACHAT_REL_BASELINE,BAISSE_ACHATS_PCT,GAP_KG,REGIME
33,2025-09-07,281695.8,390322,486330.90,0.58,42.08,-108626.2,DESTOCKAGE
34,2025-10-05,265897.1,333766,414223.35,0.64,35.81,-67868.9,DESTOCKAGE


## 15. Focus 2025–2026

In [15]:
import os

# 1. Filtrage des données
focus_recent = df_compare[df_compare["DATE"] >= "2025-01-01"][
    [
        "DATE",
        "SELL_IN_KG",
        "SELL_OUT_KG",
        "GAP_KG",
        "RATIO_IN_OUT",
        "REGIME",
        "BAISSE_ACHATS_FORTE",
        "BAISSE_ACHATS_PCT",
        "STOCK_PROXY_CUM_KG",
    ]
].copy()

display(focus_recent.round(2))

# 2. Préparation de la colonne de taille pour Plotly
focus_recent["SIZE_BUBBLE"] = focus_recent["GAP_KG"].abs().clip(lower=1)

# 3. Création du graphique à bulles
fig = px.scatter(
    focus_recent,
    x="DATE",
    y="RATIO_IN_OUT",
    color="REGIME",
    size="SIZE_BUBBLE",  # Passer le nom de la colonne en string
    hover_data={
        "SIZE_BUBBLE": False,  # Masquer cette colonne technique du survol
        "SELL_IN_KG": ":,.0f",
        "SELL_OUT_KG": ":,.0f",
        "GAP_KG": ":,.0f",
        "BAISSE_ACHATS_PCT": ":.1f",
    },
    title="CARREFOUR — Régimes d’approvisionnement 2025–2026 (kg)",
)

# 4. Ajout des lignes de repère
fig.add_hline(
    y=upper_stock,
    line_dash="dash",
    annotation_text=f"Accumulation > {upper_stock:.2f}",
)
fig.add_hline(y=1, line_dash="dot", annotation_text="Équilibre")
fig.add_hline(
    y=lower_stock,
    line_dash="dash",
    annotation_text=f"Déstockage < {lower_stock:.2f}",
)

fig.update_layout(template="plotly_white", height=560)

fig.show()

# 5. Sauvegarde de l'image pour Streamlit
os.makedirs("images", exist_ok=True)
fig.write_image("images/regimes_2025_2026.png", width=1200, height=600, scale=2)

C:\Users\andrea.leylavergne\AppData\Local\Temp\ipykernel_7088\3505754213.py:18: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(focus_recent.round(2))


,DATE,SELL_IN_KG,SELL_OUT_KG,GAP_KG,RATIO_IN_OUT,REGIME,BAISSE_ACHATS_FORTE,BAISSE_ACHATS_PCT,STOCK_PROXY_CUM_KG
25,2025-01-26,275670.95,289673,-14002.05,0.95,EQUILIBRE,False,23.80,407445.65
26,2025-02-23,294981.30,307044,-12062.70,0.96,EQUILIBRE,False,18.47,395382.95
27,2025-03-23,430161.30,306851,123310.30,1.40,ACCUMULATION_STOCK,False,-45.83,518693.25
28,2025-04-20,436176.70,361018,75158.70,1.21,ACCUMULATION_STOCK,False,-47.87,593851.95
29,2025-05-18,359271.30,421321,-62049.70,0.85,EQUILIBRE,False,16.48,531802.25
30,2025-06-15,512413.70,391957,120456.70,1.31,ACCUMULATION_STOCK,False,-19.12,652258.95
31,2025-07-13,414223.35,417723,-3499.65,0.99,EQUILIBRE,False,5.03,648759.30
32,2025-08-10,486330.90,417621,68709.90,1.16,ACCUMULATION_STOCK,False,-17.41,717469.20
33,2025-09-07,281695.80,390322,-108626.20,0.72,DESTOCKAGE,True,42.08,608843.00
34,2025-10-05,265897.10,333766,-67868.90,0.80,DESTOCKAGE,True,35.81,540974.10


# PARTIE F — Analyse par EAN et extensibilité

## 16. Quels EAN expliquent les écarts ?

Avec un Sell-Out multi-EAN complet, cette cellule classe automatiquement les contributeurs positifs et négatifs. Avec le fichier actuel, un seul EAN peut apparaître.

In [16]:
product_contrib = df_product.copy()
product_contrib['ABS_GAP_KG'] = (
    product_contrib['GAP_KG'].abs()
)

top_contributors = (
    product_contrib
    .sort_values(
        ['DATE','ABS_GAP_KG'],
        ascending=[True,False]
    )
    .groupby('DATE', group_keys=False)
    .head(10)
    [[
        'DATE','EAN','RECETTE',
        'SELL_IN_KG','SELL_OUT_KG',
        'GAP_KG','RATIO_IN_OUT','REGIME'
    ]]
)

display(top_contributors.tail(30).round(2))


C:\Users\andrea.leylavergne\AppData\Local\Temp\ipykernel_7088\1581821851.py:21: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(top_contributors.tail(30).round(2))


,DATE,EAN,RECETTE,SELL_IN_KG,SELL_OUT_KG,GAP_KG,RATIO_IN_OUT,REGIME
2071,2026-05-17,3497917004226,TAJINE DE POULET & EPICES,26710.00,8670,18040.00,3.08,ACCUMULATION_STOCK
1539,2026-05-17,3497917002741,MOZZARELLA & PESTO,34157.50,24812,9345.50,1.38,ACCUMULATION_STOCK
1357,2026-05-17,3497917002178,FROMAGE DU JURA,23072.00,14988,8084.00,1.54,ACCUMULATION_STOCK
422,2026-05-17,3497911219251,SEL DE GUERANDE,19860.00,13117,6743.00,1.51,ACCUMULATION_STOCK
334,2026-05-17,3497911131126,POULET BRAISE,21432.00,15774,5658.00,1.36,ACCUMULATION_STOCK
2097,2026-05-17,3497917004325,SAUCE BBQ,710.00,6309,-5599.00,0.11,DESTOCKAGE
550,2026-05-17,3497917000075,SEL DE GUERANDE,18400.00,13221,5179.00,1.39,ACCUMULATION_STOCK
2083,2026-05-17,3497917004257,BBQ & JALAPENO,708.75,5775,-5066.25,0.12,DESTOCKAGE
1007,2026-05-17,3497917000938,CHEDDAR & OIGNONS DE ROSCOFF,9490.00,5249,4241.00,1.81,ACCUMULATION_STOCK
128,2026-05-17,3497911105127,SEL ET VINAIGRE,24160.00,28163,-4003.00,0.86,EQUILIBRE


## 17. Vérifier les EAN Sell-In non couverts par Nielsen

Cette table indique les références CARREFOUR pour lesquelles nous disposons de livraisons mais pas de Sell-Out dans le fichier actuel. Elle permet de demander le bon export Nielsen.

In [17]:
missing_out_eans = (
    df_in_raw[~df_in_raw['EAN'].isin(upc_nielsen)]
    .groupby('EAN', as_index=False)
    .agg(
        SELL_IN_KG_TOTAL=('SELL_IN_KG','sum'),
        SELL_IN_UVC_TOTAL=('SELL_IN_UVC','sum'),
        PREMIERE_DATE=('DATE','min'),
        DERNIERE_DATE=('DATE','max')
    )
    .sort_values('SELL_IN_KG_TOTAL', ascending=False)
)

display(missing_out_eans.head(30).round(0))
print('Nombre de CODE EAN CARREFOUR absents des UPC Nielsen :', len(missing_out_eans))


C:\Users\andrea.leylavergne\AppData\Local\Temp\ipykernel_7088\2918575022.py:13: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(missing_out_eans.head(30).round(0))


,EAN,SELL_IN_KG_TOTAL,SELL_IN_UVC_TOTAL,PREMIERE_DATE,DERNIERE_DATE
5,3497917000150,180845.0,452112,2023-01-22,2026-07-12
11,3497917002260,99633.0,362302,2023-03-19,2026-07-12
0,3497911101372,94648.0,344176,2023-03-12,2026-07-12
8,3497917000525,59131.0,215022,2023-03-12,2026-07-12
4,3497917000082,46712.0,169862,2023-03-12,2026-07-12
1,3497911219282,42544.0,154704,2023-03-12,2026-07-12
28,3760326670086,26375.0,202888,2023-01-22,2026-07-12
23,3497917004431,18859.0,188592,2026-03-22,2026-07-05
3,3497917000068,18538.0,46344,2023-03-12,2026-07-12
10,3497917001294,18306.0,36612,2023-10-01,2025-06-29


Nombre de CODE EAN CARREFOUR absents des UPC Nielsen : 30


# PARTIE G — Décalage achat → vente consommateur

## 18. Tester le lag uniquement sur le périmètre comparable

La corrélation est calculée sur les séries UVC agrégées des EAN couverts, pour des décalages de 0 à 8 semaines.

In [18]:
import os

# 1. Traitement des données et calcul des lags
sell_in_weekly_matched = (
    df_in_raw[df_in_raw["EAN"].isin(matched_eans)]
    .groupby("DATE", as_index=False)
    .agg(SELL_IN_UVC=("SELL_IN_UVC", "sum"))
    .sort_values("DATE")
)

sell_out_4w_matched = (
    sell_out_product_4w.groupby("DATE", as_index=False)
    .agg(SELL_OUT_KG=("SELL_OUT_KG", "sum"))
    .sort_values("DATE")
)


def sell_in_window_for_lag(end_date, lag_weeks):
    shifted_end = end_date - pd.Timedelta(weeks=lag_weeks)
    expected = [shifted_end - pd.Timedelta(weeks=i) for i in range(4)]
    w = sell_in_weekly_matched[sell_in_weekly_matched["DATE"].isin(expected)]
    if w["DATE"].nunique() != 4:
        return np.nan
    return w["SELL_IN_UVC"].sum()


lag_rows = []
for lag in range(MAX_LAG_WEEKS + 1):
    temp = sell_out_4w_matched.copy()
    temp["SELL_IN_LAG_UVC"] = temp["DATE"].apply(
        lambda d: sell_in_window_for_lag(d, lag)
    )
    temp = temp.dropna()
    corr = (
        temp["SELL_IN_LAG_UVC"].corr(temp["SELL_OUT_KG"])
        if len(temp) >= 3
        else np.nan
    )
    lag_rows.append(
        {"LAG_SEMAINES": lag, "CORRELATION": corr, "NB_PERIODES": len(temp)}
    )

df_lags = pd.DataFrame(lag_rows)
display(df_lags.round(3))

best_lag = (
    df_lags.loc[df_lags["CORRELATION"].idxmax()]
    if df_lags["CORRELATION"].notna().any()
    else None
)
if best_lag is not None:
    print(
        f"Meilleur lag observé : {int(best_lag['LAG_SEMAINES'])} semaine(s), corrélation = {best_lag['CORRELATION']:.3f}"
    )

# 2. Création du graphique
fig = px.line(
    df_lags,
    x="LAG_SEMAINES",
    y="CORRELATION",
    markers=True,
    title="CARREFOUR — Corrélation Sell-In / Sell-Out selon le décalage",
)
fig.update_layout(template="plotly_white", height=480)
fig.show()

# 3. Sauvegarde sous un nom d'image explicite
os.makedirs("images", exist_ok=True)
fig.write_image(
    "images/correlation_lags.png", width=1200, height=600, scale=2
)

,LAG_SEMAINES,CORRELATION,NB_PERIODES
0,0,0.782,45
1,1,0.784,45
2,2,0.891,45
3,3,0.901,44
4,4,0.859,44
5,5,0.787,44
6,6,0.837,44
7,7,0.815,43
8,8,0.773,43


Meilleur lag observé : 3 semaine(s), corrélation = 0.901


# PARTIE H — Export et synthèse

## 19. Exporter les résultats

In [19]:
OUTPUT_DIR = Path.cwd() / 'outputs_carrefour_ean'
OUTPUT_DIR.mkdir(exist_ok=True)

df_product.to_csv(OUTPUT_DIR/'carrefour_ean_sellin_sellout_4w.csv', index=False, encoding='utf-8-sig')
df_compare.to_csv(OUTPUT_DIR/'carrefour_perimetre_couvert_4w.csv', index=False, encoding='utf-8-sig')
episodes.to_csv(OUTPUT_DIR/'carrefour_episodes_stock.csv', index=False, encoding='utf-8-sig')
purchase_drops.to_csv(OUTPUT_DIR/'carrefour_baisses_achats.csv', index=False, encoding='utf-8-sig')
transitions.to_csv(OUTPUT_DIR/'carrefour_transitions_achat_stock.csv', index=False, encoding='utf-8-sig')
missing_out_eans.to_csv(OUTPUT_DIR/'carrefour_ean_sans_sellout.csv', index=False, encoding='utf-8-sig')
df_lags.to_csv(OUTPUT_DIR/'carrefour_lags.csv', index=False, encoding='utf-8-sig')

print('Exports créés dans :', OUTPUT_DIR.resolve())

Exports créés dans : C:\Users\andrea.leylavergne\IA Projects\SELL_IN_SELL_OUT\outputs_carrefour_ean


## 20. Synthèse automatique de lecture métier

In [20]:
print('=== SYNTHÈSE CARREFOUR — KG UNIQUEMENT ===')
print(f"CODE EAN dans CARREFOUR_IN : {len(ean_sell_in)}")
print(f"UPC dans Nielsen           : {len(upc_nielsen)}")
print(f"Références communes        : {len(matched_eans)}")
print()

print('Règle utilisée :')
print('  CARREFOUR_IN[CODE EAN] ∩ Nielsen[UPC]')
print('  -> seuls les codes communs sont comparés')
print()

print('Unité de calcul : KG')
print('  Sell-In  = CARREFOUR_IN[CARREFOUR]')
print('  Sell-Out = Nielsen[CARREFOUR]')
print()

if not transitions.empty:
    print('Transitions candidates achat → utilisation du stock :')
    for _, r in transitions.iterrows():
        print(
            f" - {r['DATE']:%d/%m/%Y} | "
            f"Sell-In {r['SELL_IN_KG']:,.0f} kg | "
            f"Sell-Out {r['SELL_OUT_KG']:,.0f} kg | "
            f"ratio {r['RATIO_IN_OUT']:.2f}"
        )
else:
    print('Aucune transition détectée avec les seuils actuels.')


=== SYNTHÈSE CARREFOUR — KG UNIQUEMENT ===
CODE EAN dans CARREFOUR_IN : 96
UPC dans Nielsen           : 95
Références communes        : 66

Règle utilisée :
  CARREFOUR_IN[CODE EAN] ∩ Nielsen[UPC]
  -> seuls les codes communs sont comparés

Unité de calcul : KG
  Sell-In  = CARREFOUR_IN[CARREFOUR]
  Sell-Out = Nielsen[CARREFOUR]

Transitions candidates achat → utilisation du stock :
 - 31/12/2023 | Sell-In 130,254 kg | Sell-Out 232,694 kg | ratio 0.56
 - 07/09/2025 | Sell-In 281,696 kg | Sell-Out 390,322 kg | ratio 0.72
 - 05/10/2025 | Sell-In 265,897 kg | Sell-Out 333,766 kg | ratio 0.80


# Conclusion métier

Le périmètre est construit par intersection stricte :

`CARREFOUR_IN["CODE EAN"] ∩ Nielsen["UPC"]`

Puis, pour chaque EAN commun et chaque période Nielsen de 4 semaines :

- `SELL_IN_KG` = somme de la colonne `CARREFOUR` dans CARREFOUR_IN ;
- `SELL_OUT_KG` = valeur de la colonne `CARREFOUR` dans Nielsen ;
- `GAP_KG = SELL_IN_KG - SELL_OUT_KG` ;
- `RATIO_IN_OUT = SELL_IN_KG / SELL_OUT_KG`.

Lecture :
- ratio > 1 : les achats dépassent les ventes → accumulation potentielle de stock ;
- ratio < 1 : les ventes dépassent les achats → consommation du stock / déstockage ;
- les seuils de détection sont appliqués **uniquement sur les kg**.

La colonne `CARREFOUR en UVC` du Sell-In n'est utilisée qu'à titre informatif.


jupyter nbconvert --to html mon_notebook.ipynb

In [21]:
!jupyter nbconvert --to html --embed-images --no-input CARREFOUR_sellin_sellout_EAN_v4_KG_corrige.ipynb

[NbConvertApp] Converting notebook CARREFOUR_sellin_sellout_EAN_v4_KG_corrige.ipynb to html
C:\Users\andrea.leylavergne\IA Projects\SELL_IN_SELL_OUT\new_env\share\jupyter\nbconvert\templates\base\display_priority.j2:32: UserWarning: Your element with mimetype(s) dict_keys(['application/vnd.plotly.v1+json']) is not able to be represented.
  {%- elif type == 'text/vnd.mermaid' -%}
[NbConvertApp] Writing 347622 bytes to CARREFOUR_sellin_sellout_EAN_v4_KG_corrige.html
